# 01 · Chuẩn bị dữ liệu

Chọn `REBUILD = False` để kiểm tra bộ dữ liệu v4 đã duyệt và tái lập lần huấn luyện hiện tại. Gắn output 01_DATA v4 vào Kaggle Input. Chọn `REBUILD = True` để chạy quy trình tải và xử lý từ đầu.

Khi xây lại dữ liệu, xem danh sách CLIP và background trước khi xuất. Lưu quyết định `giu` hoặc `loai` theo `image_id` vào `decisions.csv` rồi chạy lại cell xuất. Các tập kiểm tra cần được giữ tách biệt với tập huấn luyện.


### Đối chiếu yêu cầu dữ liệu

- Mỗi lớp động vật tải tối đa **600 train, 150 validation, 150 test** từ split Open Images tương ứng. **20–40%** là dự trù hao hụt, cần số liệu từng bước để đo tỷ lệ thực tế. Số lượng đích khi xuất là **200/50/50**.
- Bước crop dùng box lớn nhất: dưới **30%** diện tích thì crop và nới lề **15%** mỗi phía. Sau đó loại ảnh có cạnh ngắn dưới **128 px**.
- Lượt Kaggle đã lưu dùng `REBUILD=False` để kiểm tra bộ dữ liệu đã duyệt. Cell cuối kiểm tra ngưỡng 128 px trực tiếp trên ảnh trong manifest. Bảng phía trên ghi rõ lớp và split còn thiếu ảnh.


In [1]:
# Chọn chế độ chạy. Dùng bộ dữ liệu đã duyệt để tái lập kết quả; bật REBUILD khi cần làm lại từ nguồn.
from pathlib import Path
import shutil

REBUILD = False
PUBLISHED = Path('/kaggle/input/notebooks/ngaqyh/01-data/animal_data')
ROOT = Path('/kaggle/working/animal_data')
ROOT.mkdir(parents=True, exist_ok=True)
if REBUILD and not (ROOT / 'decisions.csv').exists() and (PUBLISHED / 'decisions.csv').is_file():
    shutil.copy2(PUBLISHED / 'decisions.csv', ROOT / 'decisions.csv')
print('Chế độ:', 'Xây lại dữ liệu' if REBUILD else 'Kiểm tra dữ liệu v4 đã duyệt')


Chế độ: Kiểm tra dữ liệu v4 đã duyệt


## Cài thư viện, thiết lập cache và seed

In [2]:
# Chuẩn bị thư viện, đường dẫn làm việc và seed. Seed giúp lặp lại các bước lấy mẫu.
if REBUILD:
    import sys
    import subprocess

    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'fiftyone', 'open_clip_torch', 'imagehash', 'pillow', 'pandas'
    ])

    import os
    import json
    import random
    import shutil
    import math
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import torch
    import imagehash
    from PIL import Image

    # Trên Kaggle dùng /kaggle/working thay cho Google Drive của Colab.
    ROOT = Path('/kaggle/working/animal_data')
    ZOO_DIR = ROOT / 'fiftyone_zoo'
    PROCESSED_DIR = ROOT / 'processed'
    FINAL_DIR = ROOT / 'dataset_final'

    for folder in [ROOT, ZOO_DIR, PROCESSED_DIR]:
        folder.mkdir(parents=True, exist_ok=True)

    os.environ['FIFTYONE_DATASET_ZOO_DIR'] = str(ZOO_DIR)
    os.environ['FIFTYONE_DATABASE_DIR'] = str(ROOT / 'fiftyone_db')

    import fiftyone as fo
    import fiftyone.zoo as foz
    from fiftyone import ViewField as F

    fo.config.dataset_zoo_dir = str(ZOO_DIR)

    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    CLASSES = ['Dog', 'Cat', 'Horse', 'Sheep', 'Goat',
               'Deer', 'Tiger', 'Penguin', 'Chicken']
    ALL_CLASSES = CLASSES + ['background']
    RAW_COUNTS = {'train': 600, 'validation': 150, 'test': 150}
    FINAL_COUNTS = {'train': 200, 'val': 50, 'test': 50}
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Nếu đã Add Input dataset cũ, dùng lại ảnh và annotation đã tải.
    CACHE_DIR = Path('/kaggle/input/datasets/ngaqyh/gk-cnm-project-zip/fiftyone_zoo')
    if CACHE_DIR.is_dir():
        print('Đang chép cache, bước này có thể mất vài phút.', flush=True)
        shutil.copytree(CACHE_DIR, ZOO_DIR, dirs_exist_ok=True)

    print('Thiết bị:', DEVICE)
    print('Kết quả lưu tại:', ROOT)


## Kiểm tra lớp và bounding box bằng mẫu nhỏ

In [3]:
# Tải thử để xác nhận tên lớp, trường bounding box và thuộc tính Open Images. Giữ cả nhãn ngoài danh sách để lọc đúng ở bước sau.
if REBUILD:
    # Tìm field chứa bounding box.
    def find_box_field(dataset):
        for name, field in dataset.get_field_schema().items():
            if getattr(field, "document_type", None) == fo.Detections:
                return name
        raise ValueError("Không tìm thấy field box")


    # Tải một lớp; giữ các nhãn khác để lọc ảnh nhiều loài sau này.
    def load_class(class_name, split, limit):
        return foz.load_zoo_dataset(
            "open-images-v7",
            split=split,
            label_types=["detections"],
            classes=[class_name],
            max_samples=limit,
            only_matching=False,
            shuffle=True,
            seed=SEED,
            dataset_name=f"oi_{class_name}_{split}_{limit}",
        )


    # Tải thử 20 ảnh cho từng lớp.
    for class_name in CLASSES:
        dataset = load_class(class_name, "validation", 20)
        assert len(dataset) > 0, f"Không có ảnh lớp {class_name}"

        box_field = find_box_field(dataset)
        labels = dataset.count_values(box_field + ".detections.label")
        assert class_name in labels

        sample = dataset.first()
        image_id = Path(sample.filepath).stem

        print("\nLớp:", class_name, "| Số ảnh:", len(dataset))
        print("Field box:", box_field)
        print("ID ảnh gốc:", image_id)
        print(dataset.get_field_schema())
        print(sample)

## Tải ảnh riêng theo lớp và split Open Images

In [4]:
# Tải từng lớp trong từng split gốc. ID Open Images dùng để nhận diện cùng một ảnh xuất hiện khi tải nhiều lớp.
if REBUILD:
    raw = fo.Dataset()
    seen_ids = {}

    for official_split, limit in RAW_COUNTS.items():
        split = 'val' if official_split == 'validation' else official_split

        for class_name in CLASSES:
            # Nếu thiếu lớp/split nào, tăng limit riêng cho cặp đó ở đây.
            part = load_class(class_name, official_split, limit)
            box_field = find_box_field(part)
            print(class_name, official_split, ':', len(part), 'ảnh')

            for sample in part.iter_samples():
                image_id = Path(sample.filepath).stem

                if image_id in seen_ids:
                    assert seen_ids[image_id] == official_split, 'ID bị trùng chéo split'
                    continue

                new_sample = fo.Sample(
                    filepath=sample.filepath,
                    original_id=image_id,
                    official_split=official_split,
                    split=split,
                    source='OpenImages-v7',
                    tags=[split],
                )
                new_sample['detections'] = sample[box_field].copy()
                raw.add_sample(new_sample)
                seen_ids[image_id] = official_split

    print('Số ảnh gốc không lặp ID:', len(raw))


## Lọc box và gắn nhãn cartoon

In [5]:
# Bỏ box nhóm, đếm lớp mục tiêu phân biệt trên mỗi ảnh và giữ ảnh có đúng một lớp mục tiêu. Tách ảnh depiction để đánh giá riêng.
if REBUILD:
    without_group = raw.filter_labels(
        'detections', F('IsGroupOf') != True, only_matches=False
    )
    clean = fo.Dataset()

    for sample in without_group.iter_samples():
        boxes = sample.detections.detections

        # Đếm loài trước khi bỏ các nhãn khác.
        species = {box.label for box in boxes if box.label in CLASSES}
        if len(species) != 1:
            continue

        target_boxes = [box.copy() for box in boxes if box.label in CLASSES]
        is_depiction = any(box.IsDepiction for box in target_boxes)

        new_sample = fo.Sample(
            filepath=sample.filepath,
            original_id=sample.original_id,
            official_split=sample.official_split,
            split=sample.split,
            source=sample.source,
            label=list(species)[0],
            depiction=is_depiction,
            tags=[sample.split],
        )
        new_sample['detections'] = fo.Detections(detections=target_boxes)
        if is_depiction:
            new_sample.tags.append('depiction')
        clean.add_sample(new_sample)

    # IsOccluded và IsTruncated được giữ lại, không dùng để loại ảnh.
    print(clean.count_values('detections.detections.label'))


## Lọc ảnh trùng bằng pHash

In [6]:
# pHash mô tả bố cục ảnh bằng chuỗi bit. So khoảng cách Hamming để phát hiện ảnh gần trùng; ưu tiên giữ train khi trùng giữa các split.
if REBUILD:
    PHASH_THRESHOLD = 6


    def mark_duplicates(dataset, path_field, hash_field):
        # Xét train trước để khi trùng chéo split luôn giữ train.
        priority = {'train': 0, 'val': 1, 'test': 2}
        samples = sorted(
            dataset.iter_samples(),
            key=lambda sample: (priority[sample.split], sample.original_id),
        )
        kept = []
        pairs = []

        for sample in samples:
            with Image.open(sample[path_field]) as image:
                current_hash = imagehash.phash(image.convert('RGB'))

            sample[hash_field] = str(current_hash)
            sample['duplicate'] = False

            for kept_id, kept_split, kept_hash in kept:
                # imagehash định nghĩa phép trừ hai hash là khoảng cách Hamming.
                distance = current_hash - kept_hash
                if distance <= PHASH_THRESHOLD:
                    sample['duplicate'] = True
                    if 'duplicate' not in sample.tags:
                        sample.tags.append('duplicate')
                    pairs.append({
                        'image_id': sample.original_id,
                        'keep_id': kept_id,
                        'split': sample.split,
                        'keep_split': kept_split,
                        'distance': distance,
                        'hash_field': hash_field,
                    })
                    break

            if not sample.duplicate:
                kept.append((sample.original_id, sample.split, current_hash))
            sample.save()

        return pairs


    duplicate_pairs = mark_duplicates(clean, 'filepath', 'phash')
    data = clean.match(F('duplicate') == False)
    print('Còn lại sau pHash:', len(data))
    pd.DataFrame(duplicate_pairs).to_csv(ROOT / 'duplicates.csv', index=False)


## Crop hoặc giữ ảnh theo diện tích box

In [7]:
# Dùng box lớn nhất làm vùng tham chiếu, quyết định crop hoặc giữ nguyên. Lưu đường dẫn ảnh đã xử lý cùng ID gốc để truy vết.
if REBUILD:
    for sample in data.iter_samples():
        largest_box = max(
            sample.detections.detections,
            key=lambda box: box.bounding_box[2] * box.bounding_box[3],
        )
        x, y, w, h = largest_box.bounding_box

        with Image.open(sample.filepath) as original:
            image = original.convert('RGB')
            width, height = image.size
            crop_box = (0, 0, width, height)

            if w * h < 0.30:
                # Tọa độ box là tương đối; chuyển sang pixel và nới lề 15% mỗi cạnh.
                crop_box = (
                    max(0, math.floor((x - 0.15 * w) * width)),
                    max(0, math.floor((y - 0.15 * h) * height)),
                    min(width, math.ceil((x + 1.15 * w) * width)),
                    min(height, math.ceil((y + 1.15 * h) * height)),
                )
                image = image.crop(crop_box)

            sample['too_small'] = min(image.size) < 128
            if not sample.too_small:
                output_path = PROCESSED_DIR / f'{sample.original_id}.jpg'
                image.save(output_path, quality=95)
                sample['processed_path'] = str(output_path)
                sample['crop_box_pixels'] = list(crop_box)
            sample.save()

    data = clean.match((F('duplicate') == False) & (F('too_small') == False))
    print('Còn lại sau crop và lọc kích thước:', len(data))


## CLIP đánh dấu ảnh cần duyệt

In [8]:
# CLIP so ảnh với các mô tả bằng chữ. Bất đồng với nhãn gốc đưa ảnh vào danh sách duyệt thủ công.
if REBUILD:
    import open_clip

    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai', device=DEVICE
    )
    clip_model.eval()
    tokenizer = open_clip.get_tokenizer('ViT-B-32')
    prompts = [f'a photo of a {class_name.lower()}' for class_name in CLASSES]

    with torch.inference_mode():
        text_features = clip_model.encode_text(tokenizer(prompts).to(DEVICE))
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    clip_queue = []
    for sample in data.iter_samples():
        with Image.open(sample.processed_path) as image:
            image_tensor = clip_preprocess(image.convert('RGB')).unsqueeze(0).to(DEVICE)

        with torch.inference_mode():
            image_features = clip_model.encode_image(image_tensor)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            probabilities = (
                clip_model.logit_scale.exp() * image_features @ text_features.T
            ).softmax(dim=-1)[0]

        prediction = CLASSES[int(probabilities.argmax())]
        sample['clip_prediction'] = fo.Classification(
            label=prediction, confidence=float(probabilities.max())
        )
        sample['clip_disagree'] = prediction != sample.label

        if sample.clip_disagree:
            sample.tags.append('review_clip')
            clip_queue.append({
                'image_id': sample.original_id,
                'label': sample.label,
                'prediction': prediction,
                'filepath': sample.processed_path,
            })
        sample.save()

    pd.DataFrame(clip_queue).to_csv(ROOT / 'clip_queue.csv', index=False)
    review_view = data.match(F('clip_disagree') == True)

    # Trong App, xem ảnh rồi gắn tag giu hoặc loai. Không tự loại vì CLIP bất đồng.
    print('Ảnh CLIP cần duyệt:', len(review_view))
    del clip_model
    torch.cuda.empty_cache()


## Tạo lớp background và rà ảnh trùng

In [9]:
# Tạo background từ đồ vật và động vật ngoài danh sách. Kiểm tra nhãn động vật, ảnh trùng và đưa ảnh nền vào hàng đợi duyệt.
if REBUILD:
    OBJECT_CLASSES = ['Chair', 'Table', 'Building', 'Tree', 'Car']
    OTHER_ANIMALS = ['Lion', 'Leopard', 'Pig', 'Fox', 'Rabbit', 'Duck', 'Goose']

    hierarchy_path = ZOO_DIR / 'open-images-v7/validation/metadata/hierarchy.json'
    hierarchy = json.loads(hierarchy_path.read_text(encoding='utf-8'))


    def descendants(node):
        names = {node['LabelName']}
        for child in node.get('Subcategory', []):
            names.update(descendants(child))
        return names


    def find_animals(node):
        if node['LabelName'] == 'Animal':
            return descendants(node)
        for child in node.get('Subcategory', []):
            result = find_animals(child)
            if result:
                return result
        return set()


    # Trả về None nếu dataset không có field box.
    def find_box_field(dataset):
        for name, field in dataset.get_field_schema().items():
            if getattr(field, 'document_type', None) == fo.Detections:
                return name

        if len(dataset) > 0:
            sample = dataset.first()
            for name in dataset.get_field_schema():
                if isinstance(sample[name], fo.Detections):
                    return name
        return None


    animal_labels = find_animals(hierarchy)
    assert len(animal_labels) > 20, 'Cần kiểm tra lại hierarchy động vật'

    for official_split, limit in [('train', 80), ('validation', 30), ('test', 30)]:
        split = 'val' if official_split == 'validation' else official_split

        for group, classes in [
            ('object', OBJECT_CLASSES),
            ('hard_negative', OTHER_ANIMALS),
        ]:
            for class_name in classes:
                part = load_class(class_name, official_split, limit)
                box_field = find_box_field(part)

                # Bỏ qua nguồn thiếu annotation, không coi là background sạch.
                if box_field is None:
                    print('BỎ QUA:', class_name, official_split,
                          '| Số ảnh:', len(part), '| Không có box')
                    continue

                print('Đang xử lý:', class_name, official_split)

                for sample in part.iter_samples():
                    detections = sample[box_field]
                    if detections is None or not detections.detections:
                        continue

                    image_id = Path(sample.filepath).stem
                    labels = {box.label for box in detections.detections}

                    if image_id in seen_ids or labels.intersection(CLASSES):
                        continue
                    if group == 'object' and labels.intersection(animal_labels):
                        continue

                    with Image.open(sample.filepath) as original:
                        image = original.convert('RGB')
                        if min(image.size) < 128:
                            continue
                        output_path = PROCESSED_DIR / f'{image_id}.jpg'
                        image.save(output_path, quality=95)

                    background = fo.Sample(
                        filepath=sample.filepath,
                        processed_path=str(output_path),
                        original_id=image_id,
                        official_split=official_split,
                        split=split,
                        label='background',
                        source='OpenImages-v7',
                        background_kind=group,
                        depiction=False,
                        duplicate=False,
                        too_small=False,
                        clip_disagree=False,
                        tags=[split, 'review_background'],
                    )
                    clean.add_sample(background)
                    seen_ids[image_id] = official_split

    # Gồm cả background đã thêm từ lần chạy trước.
    background_queue = [
        {
            'image_id': sample.original_id,
            'filepath': sample.processed_path,
        }
        for sample in clean.match(F('label') == 'background').iter_samples()
    ]
    pd.DataFrame(background_queue).to_csv(
        ROOT / 'background_queue.csv', index=False
    )

    # Rà trùng sau khi thêm background: ảnh gốc rồi ảnh crop.
    data = clean.match((F('duplicate') == False) & (F('too_small') == False))
    duplicate_pairs += mark_duplicates(data, 'filepath', 'phash')

    data = clean.match((F('duplicate') == False) & (F('too_small') == False))
    duplicate_pairs += mark_duplicates(data, 'processed_path', 'processed_phash')

    data = clean.match((F('duplicate') == False) & (F('too_small') == False))
    pd.DataFrame(duplicate_pairs).to_csv(ROOT / 'duplicates.csv', index=False)

    print('Background còn lại:', len(data.match(F('label') == 'background')))
    print('Danh sách cần duyệt:', ROOT / 'background_queue.csv')

## Xuất dữ liệu đã duyệt

Cell này kiểm tra danh sách ảnh chờ duyệt rồi xuất ảnh, ID gốc và giấy phép.

In [10]:
# Áp dụng quyết định giữ/loại, thống kê theo lớp và split, rồi xuất thư mục ảnh và CSV manifest cho Notebook 02.
if REBUILD:
    # Cell 9 - Xuất dữ liệu đã duyệt và ghi số lượng thực tế
    review_file = ROOT / 'decisions.csv'
    if review_file.is_file():
        review = pd.read_csv(review_file, dtype={'image_id': str})
    else:
        review = pd.DataFrame(columns=['image_id', 'decision'])
    assert review.decision.isin(['giu', 'loai']).all() and review.image_id.is_unique
    choices = dict(zip(review.image_id, review.decision))
    rows, pending = [], []
    counts = data.count_values('label')
    removed = dict.fromkeys(ALL_CLASSES, 0)
    for s in data.iter_samples():
        decision = choices.get(s.original_id)
        if decision and decision not in s.tags:
            s.tags.append(decision)
            s.save()
        if 'loai' in s.tags:
            removed[s.label] += 1
            continue
        if (s.clip_disagree or s.label == 'background') and 'giu' not in s.tags:
            pending.append(s.original_id)
            continue
        rows.append(dict(image_id=s.original_id, label=s.label, split=s.split,
                         official_split=s.official_split, source=s.source,
                         filepath=s.processed_path, depiction=s.depiction,
                         background_kind=s['background_kind'] if 'background_kind' in s.field_names else '',
                         phash=s.processed_phash))
    rates = pd.DataFrame([dict(label=c, before_review=counts.get(c, 0), removed=removed[c],
                               removal_rate=removed[c] / counts[c] if counts.get(c) else 0)
                          for c in ALL_CLASSES])
    rates.to_csv(ROOT / 'review_rates.csv', index=False)
    print('Ảnh còn chờ duyệt:', len(pending))
    pd.DataFrame({'image_id': pending, 'decision': ''}).to_csv(ROOT / 'pending_review.csv', index=False)
    if pending:
        raise RuntimeError(f'Còn {len(pending)} ảnh. Xem pending_review.csv và lưu quyết định vào decisions.csv.')
    assert rows, 'Không còn ảnh sau khi duyệt.'

    frame = pd.DataFrame(rows)
    selected, shortages = [], []
    for label in ALL_CLASSES:
        for split, goal in FINAL_COUNTS.items():
            pool = frame[(frame.label == label) & (frame.split == split)].sample(frac=1, random_state=SEED)
            if label == 'background':
                objects = pool[pool.background_kind == 'object'].head(goal // 2)
                animals = pool[pool.background_kind == 'hard_negative'].head(goal - goal // 2)
                chosen = pd.concat([objects, animals])
            elif split == 'train':
                cartoon = pool[pool.depiction].head(25)
                chosen = pd.concat([pool[~pool.depiction].head(goal - len(cartoon)), cartoon])
                if len(cartoon) < 20:
                    shortages.append(f'{label}/train cartoon: {len(cartoon)}/20')
            else:
                chosen = pool[~pool.depiction].head(goal)
            selected.append(chosen)
            if len(chosen) < goal:
                shortages.append(f'{label}/{split}: {len(chosen)}/{goal}')
        if label != 'background':
            cartoon_test = frame[(frame.label == label) & frame.depiction &
                                 (frame.official_split.isin(['test', 'validation']) | (frame.split == 'test_cartoon'))]
            cartoon_test = cartoon_test.sample(frac=1, random_state=SEED).copy()
            cartoon_test['origin_rank'] = cartoon_test.official_split.map({'test': 0, 'validation': 1, 'train': 2})
            cartoon_test = cartoon_test.sort_values('origin_rank', kind='stable').head(15)
            cartoon_test = cartoon_test.drop(columns='origin_rank').assign(split='test_cartoon')
            selected.append(cartoon_test)
            if len(cartoon_test) < 15:
                shortages.append(f'{label}/test_cartoon: {len(cartoon_test)}/15')

    final = pd.concat(selected, ignore_index=True)
    print(pd.crosstab(final.label, final.split))
    assert final.groupby('image_id').split.nunique().max() == 1, 'ID trùng chéo tập'
    for message in shortages:
        print('THIẾU:', message)
    # Số lượng val/test thực tế vẫn giữ đúng split gốc. Ghi thiếu hụt để báo cáo.
    (ROOT / 'dataset_shortages.txt').write_text('\n'.join(shortages), encoding='utf-8')
    assert final[final.split == 'train'].groupby('label').size().reindex(ALL_CLASSES).eq(200).all()
    for split, origin in [('train','train'), ('val','validation'), ('test','test')]:
        group = final[final.split == split]
        assert set(group.label) == set(ALL_CLASSES) and group.official_split.eq(origin).all()
    assert final.image_id.is_unique

    metadata = []
    wanted_ids = set(final.image_id)
    for official_split in RAW_COUNTS:
        path = ZOO_DIR / 'open-images-v7' / official_split / 'metadata/image_ids.csv'
        for chunk in pd.read_csv(path, dtype=str, usecols=['ImageID', 'License', 'OriginalURL'], chunksize=100000):
            metadata.append(chunk[chunk.ImageID.isin(wanted_ids)])
    metadata = pd.concat(metadata).drop_duplicates('ImageID').rename(columns={
        'ImageID': 'image_id', 'License': 'license', 'OriginalURL': 'original_url'})
    final = final.merge(metadata, on='image_id', how='left')
    assert final.license.notna().all(), 'Có ảnh chưa có giấy phép'

    paths = []
    for row in final.itertuples():
        path = FINAL_DIR / row.split / row.label / f'{row.image_id}.jpg'
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(row.filepath, path)
        paths.append(str(path.relative_to(ROOT)))
    for split in ['train', 'val', 'test']:
        folders = sorted(p.name for p in (FINAL_DIR / split).iterdir() if p.is_dir())
        assert folders == sorted(ALL_CLASSES), f'{split} chưa đủ 10 lớp'
    final['relpath'] = paths
    final.drop(columns='filepath').to_csv(ROOT / 'dataset_manifest.csv', index=False)
    clean.write_json(str(ROOT / 'fiftyone_dataset.json'))
    assert len(paths) == len(final) and all((ROOT / relpath).is_file() for relpath in paths)
    print('Số ảnh theo tập:', final.split.value_counts().to_dict())
    print('Đã xuất', len(final), 'ảnh và dataset_manifest.csv')

## Kiểm tra dữ liệu trước khi train

In [11]:
# Kiểm tra bộ đã xuất: các split, danh sách lớp và các file cần cho bước huấn luyện.
import csv
from collections import Counter
from IPython.display import Markdown, display

dataset = ROOT if REBUILD else PUBLISHED
with (dataset / 'dataset_manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    records = list(csv.DictReader(stream))
assert records, 'Manifest rỗng'
ids = [r['image_id'] for r in records]
assert len(ids) == len(set(ids)), 'Có ID ảnh trùng'
expected = {'train': 'train', 'val': 'validation', 'test': 'test'}
assert all(r['official_split'] == expected[r['split']] for r in records if r['split'] in expected)
assert all(r['license'] for r in records), 'Thiếu giấy phép'
missing = [r['relpath'] for r in records if not (dataset / r['relpath']).is_file()]
assert not missing, f'Thiếu {len(missing)} file ảnh. Ví dụ {missing[:3]}'
counts = Counter(r['split'] for r in records)
labels = sorted({r['label'] for r in records})
assert len(labels) == 10
assert all({r['label'] for r in records if r['split'] == split} == set(labels) for split in expected)
by_class = Counter((r['label'], r['split']) for r in records)
splits = ['train', 'val', 'test', 'test_cartoon']
rows = ['| Lớp | Train | Val | Test | Cartoon |', '| --- | ---: | ---: | ---: | ---: |']
for label in labels:
    rows.append('| ' + ' | '.join([label] + [str(by_class[label, split]) for split in splits]) + ' |')
rows.append('| **Tổng** | ' + ' | '.join(str(counts[split]) for split in splits) + ' |')
display(Markdown('\n'.join(rows)))
for filename in ['dataset_manifest.csv', 'review_rates.csv', 'dataset_shortages.txt']:
    source = dataset / filename
    if source.is_file() and source != ROOT / filename:
        shutil.copy2(source, ROOT / filename)
shortages = dataset / 'dataset_shortages.txt'
if shortages.is_file() and shortages.read_text(encoding='utf-8').strip():
    display(Markdown('**Số lượng dưới mức dự kiến**\n\n' +
                     '\n'.join('- ' + line for line in shortages.read_text(encoding='utf-8').splitlines())))
print('Đã kiểm tra', len(records), 'ảnh và split gốc. Nguồn ảnh:', dataset)


| Lớp | Train | Val | Test | Cartoon |
| --- | ---: | ---: | ---: | ---: |
| Cat | 200 | 50 | 50 | 15 |
| Chicken | 200 | 46 | 50 | 15 |
| Deer | 200 | 50 | 50 | 15 |
| Dog | 200 | 50 | 50 | 15 |
| Goat | 200 | 30 | 50 | 10 |
| Horse | 200 | 50 | 50 | 15 |
| Penguin | 200 | 31 | 50 | 15 |
| Sheep | 200 | 19 | 50 | 15 |
| Tiger | 200 | 14 | 34 | 15 |
| background | 200 | 50 | 50 | 0 |
| **Tổng** | 2000 | 390 | 484 | 130 |

**Số lượng dưới mức dự kiến**

- Sheep/val: 19/50
- Goat/val: 30/50
- Goat/test_cartoon: 10/15
- Tiger/val: 14/50
- Tiger/test: 34/50
- Penguin/val: 31/50
- Chicken/val: 46/50

Đã kiểm tra 3004 ảnh và split gốc. Nguồn ảnh: /kaggle/input/notebooks/ngaqyh/01-data/animal_data


## Kiểm tra cạnh ngắn của ảnh đã xuất

Output dưới đây chạy trên gói dữ liệu tải về từ Kaggle.

In [12]:
# Đọc kích thước ảnh đã xuất để kiểm tra ngưỡng 128 px, kể cả khi REBUILD=False.
from PIL import Image

shortest = []
for row in records:
    with Image.open(dataset / row['relpath']) as image:
        shortest.append(min(image.size))

below_limit = sum(side < 128 for side in shortest)
print(f'Đã kiểm tra {len(shortest):,} ảnh. Cạnh ngắn nhỏ nhất: {min(shortest)} px.')
print(f'Số ảnh có cạnh ngắn dưới 128 px: {below_limit}')
assert below_limit == 0, 'Bộ dữ liệu còn ảnh dưới 128 px.'


Đã kiểm tra 3,004 ảnh. Cạnh ngắn nhỏ nhất: 128 px.
Số ảnh có cạnh ngắn dưới 128 px: 0
